# Deber — Semana 3: Carga de Datos, Series/DataFrames, loc/iloc y Análisis de Texto
## Análisis de Datos (TDSD353) | Período 2026-A
## ESFOT — Escuela Politécnica Nacional


**Nombre:** Alan Javier Reyes Casierra    
**Fecha:** 25/4/2026
<br/>
---

### Objetivo de este Taller
Aplicar las técnicas aprendidas en clase para:
1. Cargar datos desde archivos **JSON**, **TXT** (CSV) y **XML** usando Pandas.
2. Detectar y documentar **problemas de calidad de datos**.
3. Unificar múltiples DataFrames con `pd.concat()` y exportar a CSV.
4. Comprender y usar **Series** y **DataFrames**.
5. Acceder a datos con **`loc`** (por etiqueta) e **`iloc`** (por posición).
6. Descargar y procesar texto desde la web con `urllib`.

### Archivos de trabajo
| Archivo | Contenido | IDs |
|---------|-----------|-----|
| `productos.json` | 20 productos de librería | 1–20 |
| `productos.txt` | 20 productos de librería (formato CSV) | 21–40 |
| `productos.xml` | 20 productos de librería | 41–60 |
| `netflix.csv` | Catálogo de Netflix (~8800 títulos) | — |

> ⚠️ Los archivos de productos contienen **errores intencionales** (`"error"`, `"N/A"`, `null`, stock negativo, campos vacíos) para ilustrar problemas reales de calidad de datos.

---

---
## PARTE A: Pipeline de datos — Inventario de Papelería

Tenemos 3 archivos con datos del inventario de una Papelería:
- `productos.json` — 20 productos (IDs 1–20)
- `productos.txt` — 20 productos (IDs 21–40, separados por coma)
- `productos.xml` — 20 productos (IDs 41–60)

Columnas compartidas: `id`, `nombre`, `precio_compra`, `categoria`, `stock`, `proveedor`, `precio_venta_publico`

> ⚠️ Los datos contienen errores intencionales (`"error"`, `"N/A"`, `null`, stock negativo, campos vacíos) para ilustrar problemas reales de calidad de datos.

### A-1. Importar librerías

| Librería | Uso |
|----------|-----|
| `pandas` | Manipulación de datos tabulares |
| `urllib.request` | Descarga de archivos desde la web |
| `collections.Counter` | Conteo de frecuencias |
| `time` | Medición de tiempos |
| `re` | Expresiones regulares para limpieza de texto |

In [88]:
import pandas as pd
from urllib import request
from collections import Counter
import time
import re

### A-2. Cargar JSON con `pd.read_json()`

**JSON** (JavaScript Object Notation): formato ligero muy común en APIs web.

```python
df = pd.read_json('archivo.json')
```

Parámetros útiles: `orient` (formato del JSON), `encoding`, `dtype` (forzar tipos).

In [89]:
df_json = pd.read_json('productos.json')
df_json.head()

,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,1,Bolígrafo BIC Azul,0.35,Escritura,500,DistribuPapel S.A.,0.75
1,2,Cuaderno Universitario 100 hojas,1.8,Cuadernos,300,Norma Ecuador,3.50
2,3,Lápiz HB Faber-Castell,0.2,Escritura,800,DistribuPapel S.A.,0.45
3,4,Resaltador Amarillo,error,Escritura,150,OfficeMax Ecuador,1.20
4,5,Carpeta Manila A4,0.15,Archivo,1000,PapelPro Cía. Ltda.,0.30


### A-3. Cargar TXT con `pd.read_csv()`

**TXT tabular** (datos separados por coma): los archivos `.txt` con formato de tabla se leen con `read_csv()` ajustando el separador.

```python
df = pd.read_csv('archivo.txt', sep=',')   # separador: coma
df = pd.read_csv('archivo.txt', sep='\t')  # separador: tabulación
```

> **Diferencia CSV vs TXT**: ambos son texto plano. La extensión `.txt` no implica un formato fijo — siempre revisa el separador antes de cargar.

In [90]:
df_txt = pd.read_csv('productos.txt')
df_txt.head()

,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,21,Sacapuntas Doble Uso,0.15,Escritura,700,DistribuPapel S.A.,0.35
1,22,Cuaderno Espiral A5 80 hojas,1.40,Cuadernos,220,Norma Ecuador,2.80
2,23,Lápiz de Color x24,2.10,Manualidades,160,Norma Ecuador,4.20
3,24,Pegamento en Barra 40g,0.55,Manualidades,430,OfficeMax Ecuador,1.05
4,25,Papel Lustre x10 colores,0.60,Papel,380,PapelPro Cía. Ltda.,1.20


### A-4. Cargar XML con `pd.read_xml()`

**XML** (eXtensible Markup Language): formato jerárquico usado en servicios web SOAP y datos gubernamentales.

```python
df = pd.read_xml('archivo.xml', xpath='.//producto')
```

El parámetro `xpath` indica qué nodos representan cada fila. 

In [91]:
#%pip install lxml

df_xml = pd.read_xml('productos.xml', xpath='.//producto')
df_xml.head()

,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,41,Tiza Blanca Caja x100,1.5,Escritura,180,Norma Ecuador,3.00
1,42,Papelógrafo Cuadriculado x50,2.8,Papel,130,PapelPro Cía. Ltda.,5.20
2,43,Organizador de Escritorio Plástico,3.6,Organización,65,OfficeMax Ecuador,6.80
3,44,Cuaderno Cosido 50 hojas,0.9,Cuadernos,texto,Norma Ecuador,1.80
4,45,Cartulina de Colores x10,0.7,Papel,320,PapelPro Cía. Ltda.,1.40


Al ejecutar la celda anterior puede existir un error: `ImportError: lxml not found, please install or use the etree parser.`  
Lo solucionamos con: `%pip install lxml`

### A-5. Verificar consistencia y detectar errores

Antes de concatenar, verificamos que los 3 DataFrames tengan las **mismas columnas** y documentamos los problemas de calidad.

Errores comunes en datos reales:
- Valores `null` / `NaN` (datos faltantes)
- Texto donde debería haber números (`"error"`, `"N/A"`, `"texto"`)
- Valores lógicamente inválidos (Ejemplo stock negativo)
- Campos vacíos (`""`)

In [92]:
print(f'Son iguales los 3 dataframes? {set(df_txt.columns) == set(df_json.columns) == set(df_xml.columns)}')

Son iguales los 3 dataframes? True


In [93]:
print('Valores vacíos por columna de cada df')
# print(f'df_xml\n{df_xml.isnull().sum()} \ndf_txt\n{df_txt.isnull().sum()} \ndf_json\n{df_json.isnull().sum()}')

dfs = {
    'json': df_json,
    'xml': df_xml,
    'txt': df_txt
}

for nombre, df in dfs.items():
    print(f'\t---{nombre.upper()}---\n{df.isnull().sum()}')

Valores vacíos por columna de cada df
	---JSON---
id                      0
nombre                  0
precio_compra           1
categoria               0
stock                   0
proveedor               1
precio_venta_publico    0
dtype: int64
	---XML---
id                      0
nombre                  0
precio_compra           2
categoria               0
stock                   0
proveedor               1
precio_venta_publico    0
dtype: int64
	---TXT---
id                      0
nombre                  0
precio_compra           2
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64


In [107]:
print(f'Valores inválidos de cada df')
for nombre, df in dfs.items():
    print(nombre.upper())
    df['stock'] = pd.to_numeric(df['stock'], errors='coerce')
    display(df[df['stock'] < 0])

Valores inválidos de cada df
JSON


,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
19,20,Crayones x12 Colores,0.95,Manualidades,-30.0,Norma Ecuador,1.8


XML


,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
16,57,Cartón Corrugado 50x70cm,0.8,Manualidades,-15.0,PapelPro Cía. Ltda.,1.60


TXT


,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico


In [95]:
for nombre, df in dfs.items():
    print(f'\t---{nombre.upper()}---')
    print((df == '').sum())

	---JSON---
id                      0
nombre                  0
precio_compra           0
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64
	---XML---
id                      0
nombre                  0
precio_compra           0
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64
	---TXT---
id                      0
nombre                  0
precio_compra           0
categoria               0
stock                   0
proveedor               0
precio_venta_publico    0
dtype: int64


### A-6. Concatenar con `pd.concat()` y exportar con el metodo to_csv()

`pd.concat()` apila DataFrames verticalmente. `ignore_index=True` reinicia el índice.

| Función | Uso |
|---------|-----|
| `pd.concat()` | Apilar DataFrames con **misma estructura** |

In [108]:
df_final = pd.concat([df_json, df_xml, df_txt], ignore_index=True)
df_final

,id,nombre,precio_compra,categoria,stock,proveedor,precio_venta_publico
0,1,Bolígrafo BIC Azul,0.35,Escritura,500.0,DistribuPapel S.A.,0.75
1,2,Cuaderno Universitario 100 hojas,1.8,Cuadernos,300.0,Norma Ecuador,3.5
2,3,Lápiz HB Faber-Castell,0.2,Escritura,800.0,DistribuPapel S.A.,0.45
3,4,Resaltador Amarillo,error,Escritura,150.0,OfficeMax Ecuador,1.2
4,5,Carpeta Manila A4,0.15,Archivo,1000.0,PapelPro Cía. Ltda.,0.3
5,6,Tijeras Escolares 13cm,0.9,Manualidades,200.0,OfficeMax Ecuador,1.8
6,7,Corrector Líquido 18ml,None,Escritura,180.0,Norma Ecuador,1.5
7,8,Goma de Borrar Blanca,0.25,Escritura,600.0,DistribuPapel S.A.,0.55
8,9,Resma Papel Bond A4 75g,3.2,Papel,250.0,PapelPro Cía. Ltda.,5.5
9,10,Marcador Permanente Negro,0.6,Escritura,400.0,OfficeMax Ecuador,1.1


Exportar el dataframe concatenado con `to_csv()` con index=False  como argumento adicional, evitar escribir el índice numérico como columna extra

In [111]:
df_final.to_csv('productos_final.csv', index=False)

---
---
## PARTE B: Series, DataFrames, `loc` e `iloc` — Catálogo de Netflix

Trabajaremos con el archivo `netflix.csv`, un dataset real con el catálogo de contenidos de Netflix.

| Columna | Descripción |
|---------|------------|
| `show_id` | ID único del título |
| `type` | `Movie` o `TV Show` |
| `title` | Nombre del título |
| `director` | Director(es) |
| `cast` | Elenco principal |
| `country` | País de producción |
| `date_added` | Fecha de añadido a Netflix |
| `release_year` | Año de estreno |
| `rating` | Clasificación de audiencia |
| `duration` | Duración (minutos o temporadas) |
| `listed_in` | Géneros/categorías |
| `description` | Sinopsis |

### B-1. Carga el dataset y realiza una primera inspección

In [124]:
netflix = pd.read_csv('netflix.csv')
print(f'Tamaño del dataset netflix: {netflix.shape}')
display(netflix.head())
netflix.info()

Tamaño del dataset netflix: (6137, 15)


,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.601,NaN
1,tm82169,Rocky,MOVIE,"When world heavyweight boxing champion, Apollo...",1976,PG,119,"['drama', 'sport']",['US'],NaN,tt0075148,8.1,588100.0,106.361,7.782
2,tm17823,Grease,MOVIE,Australian good girl Sandy and greaser Danny f...,1978,PG,110,"['romance', 'comedy']",['US'],NaN,tt0077631,7.2,283316.0,33.160,7.406
3,tm191099,The Sting,MOVIE,A novice con man teams up with an acknowledged...,1973,PG,129,"['crime', 'drama', 'comedy', 'music']",['US'],NaN,tt0070735,8.3,266738.0,24.616,8.020
4,tm69975,Rocky II,MOVIE,After Rocky goes the distance with champ Apoll...,1979,PG,119,"['drama', 'sport']",['US'],NaN,tt0079817,7.3,216307.0,75.699,7.246


<class 'pandas.DataFrame'>
RangeIndex: 6137 entries, 0 to 6136
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    6137 non-null   str    
 1   title                 6137 non-null   str    
 2   type                  6137 non-null   str    
 3   description           6114 non-null   str    
 4   release_year          6137 non-null   int64  
 5   age_certification     3394 non-null   str    
 6   runtime               6137 non-null   int64  
 7   genres                6137 non-null   str    
 8   production_countries  6137 non-null   str    
 9   seasons               2306 non-null   float64
 10  imdb_id               5741 non-null   str    
 11  imdb_score            5669 non-null   float64
 12  imdb_votes            5653 non-null   float64
 13  tmdb_popularity       6061 non-null   float64
 14  tmdb_score            5885 non-null   float64
dtypes: float64(5), int64(2), str(8)


---
### B-2. ¿Qué es una Series?

Una **Series** es un arreglo **unidimensional** etiquetado. Cada elemento tiene un **valor** y un **índice**.

```python
# Desde una lista
s = pd.Series([10, 20, 30])

# Desde un diccionario (claves = índice)
s = pd.Series({'a': 10, 'b': 20, 'c': 30})
```

| Atributo | Descripción |
|----------|------------|
| `.values` | Datos como array NumPy |
| `.index` | Etiquetas del índice |
| `.dtype` | Tipo de dato |
| `.name` | Nombre de la Series |
| `.shape` | Forma (número de elementos,) |

**Cuando accedes a UNA columna de un DataFrame, obtienes una Series:**

### Extraer una columna = una Series del csv de netflix

In [168]:
netflix['title']
title = netflix['title']
title
# title.index
# title.dtype
# title.name
# title.shape

0       Five Came Back: The Reference Films
1                                     Rocky
2                                    Grease
3                                 The Sting
4                                  Rocky II
                       ...                 
6132                          عبود في البيت
6133                                Sweetie
6134              Sommore: Queen Chandelier
6135                           All Na Vibes
6136                        Going to Heaven
Name: title, Length: 6137, dtype: str

### Operaciones comunes sobre Series

### Obten las estadísticas de la columna 'release_year', count, min, max, mean, median, std

In [167]:
netflix['release_year']
release_year = netflix['release_year']

print(f'release_year.count(): {release_year.count()}')
print(f'release_year.min(): {release_year.min()}')
print(f'release_year.max(): {release_year.max()}')
print(f'release_year.mean(): {round(release_year.mean(), 2)}')
print(f'release_year.median(): {release_year.median()}')
print(f'release_year.std(): {round(release_year.std(), 2)}')

release_year.count(): 6137
release_year.min(): 1945
release_year.max(): 2023
release_year.mean(): 2017.37
release_year.median(): 2019.0
release_year.std(): 6.6


### Obten el Top 10 países con más contenido producido

In [169]:
netflix['production_countries']
production_countries = netflix['production_countries']

production_countries.value_counts().head(10)

production_countries
['US']    1981
['IN']     633
['JP']     278
['KR']     259
['GB']     235
['ES']     177
[]         176
['FR']     121
['MX']     115
['BR']     104
Name: count, dtype: int64

### Obten el Top 10 de series/peliculas mas vistas

In [183]:
top_10 = netflix.sort_values(by='imdb_votes', ascending=False).head(10)

print(top_10[['title', 'imdb_votes']])

                                                 title  imdb_votes
201                                    The Dark Knight   2684317.0
87                                        Forrest Gump   2106826.0
186                                       Breaking Bad   1936461.0
195  The Lord of the Rings: The Fellowship of the Ring   1895545.0
209      The Lord of the Rings: The Return of the King   1865989.0
208              The Lord of the Rings: The Two Towers   1684864.0
917                                    Stranger Things   1220079.0
88                                      Reservoir Dogs   1031133.0
187                                   The Walking Dead   1013253.0
533                                   The Hunger Games    930840.0


---
### B-3. Acceso con `loc` (por etiqueta)

**`loc`** selecciona datos por **etiqueta** del índice y **nombre** de columna.

```python
df.loc[etiqueta_fila]                    # Una fila completa
df.loc[etiqueta_fila, 'columna']         # Un valor específico
df.loc[inicio:fin]                       # Rango de filas (INCLUSIVO)
df.loc[condicion]                        # Filtrado booleano
df.loc[condicion, ['col1', 'col2']]      # Filtrado + columnas
```

> Con `loc`, los rangos son **INCLUSIVOS** en ambos extremos: `loc[2:5]` incluye 2, 3, 4 **y 5**.

In [97]:
# Genera un ejemplo de Acceso a una fila completa por su etiqueta de índice


In [98]:
# Filtrado con MÚLTIPLES condiciones
# Obten el listado de Películas de Estados Unidos producidas desde 2018


In [99]:
# Filtrado con isin() para múltiples valores
# Filtra todas las filas cuyo Contenido tenga como origen Latinoamérica
paises_latam = ['Mexico', 'Argentina', 'Colombia', 'Chile', 'Brazil', 'Peru']


In [100]:
# str.contains(): filtrado por texto parcial en columna
# Títulos que contengan 'Love' en el nombre


### B-4. Ya que tienes acceso a los datos de Netflix, en base a tu curiosidad, usando pandas obten al menos 3 datos de interes adicionales, que te gustaria conocer.

---
### B-5. Acceso con `iloc` (por posición entera)

**`iloc`** selecciona datos por **posición numérica** (como índices de listas en Python).

```python
df.iloc[posicion]                          # Una fila
df.iloc[pos_fila, pos_columna]             # Un valor
df.iloc[inicio:fin]                        # Rango (EXCLUSIVO al final)
df.iloc[[0, 5, 10]]                        # Posiciones específicas
df.iloc[::n]                               # Cada n filas
```

> Con `iloc`, el rango es **EXCLUSIVO** al final: `iloc[2:5]` incluye posiciones 2, 3, 4 pero **NO 5**.

### Diferencia clave `loc` vs `iloc`:

| | `loc` | `iloc` |
|--|-------|--------|
| **Acceso** | Por **etiqueta** | Por **posición** |
| **Rango** | **Inclusivo** | **Exclusivo** |
| **Columnas** | Por nombre | Por número (0, 1, 2...) |
| **Filtrado** |  Booleano | No directo |

In [101]:
# Obten la Primera fila (posición 0) usando loc


In [102]:
# Última fila (índice negativo, como listas de Python) usando loc


In [103]:
# Valor específico: fila 10, columna 2 (title) usando loc


In [104]:
# Rango de filas (EXCLUSIVO: posiciones 2, 3, 4 — NO incluye 5) usando loc


In [105]:
# Usando loc , obten los 10 títulos más antiguos en Netflix
# Combinamos sort_values con iloc(Ejemplo base: netflix_ordenado = df_netflix.sort_values("release_year", ascending=True))
print('=== 10 títulos más antiguos en Netflix ===')


=== 10 títulos más antiguos en Netflix ===


---
---
## PARTE C: Procesamiento de Texto — *Dracula* (Bram Stoker)

En esta sección crear las cajas pertinentes a fin de realizar las siguientes actividades sobre el libro de Drácula:
1. Descargar texto desde una URL con `urllib`, crear una variable time, que contabilize el tiempo desde que se inicia la descarga.
2. Eliminar encabezados y pies de página propios de Project Gutenberg.
3. Eliminar signos de puntuación, números y caracteres especiales con `re`.
4. Procesar texto: minúsculas, tokenizar, filtrar.
5. Contar frecuencias con `collections.Counter`.
6. Mostrar el top 20 de las palabras mas frecuentes.
7. Guardar resultados en un archivo de texto(txt), donde se muestre el top 20 de palabras mas frecuentes y su cantidad de incidencias, asi como el tiempo total demorado desde que se descarga el libro, hasta que se cuenta las palabras mas frecuentes.

Usaremos **"Dracula"** de Bram Stoker, disponible en [Project Gutenberg](https://www.gutenberg.org/ebooks/345).  
https://www.gutenberg.org/cache/epub/345/pg345.txt



In [106]:
#Desarrollo del estudiante